In [58]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"] = 'true'
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

In [59]:
from langchain.chat_models import init_chat_model

google_llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
google_llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x0000014CC3675BD0>, default_metadata=(), model_kwargs={})

In [60]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str = Field(description="Title of the movie")
    year:int = Field(description="Year of release")
    rating:float = Field(description="Rating of the movie")
    genre:str = Field(description="Genre of the movie") 
    

model_with_structure = google_llm.with_structured_output(
    Movie
)
model_with_structure

RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x0000014CC3675BD0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'properties': {'title': {'description': 'Title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'Year of release', 'title': 'Year', 'type': 'integer'}, 'rating': {'description': 'Rating of the movie', 'title': 'Rating', 'type': 'number'}, 'genre': {'descripti

In [61]:
google_llm.invoke("Tell me about Avatar movie")

AIMessage(content='"Avatar" is a groundbreaking science fiction film directed by James Cameron, released in 2009. It\'s renowned for its revolutionary visual effects, immersive world-building, and a compelling, albeit somewhat familiar, story.\n\nHere\'s a breakdown of what makes "Avatar" so significant:\n\n**The Story:**\n\nThe film is set in the mid-22nd century on Pandora, a lush, vibrant, and dangerous moon inhabited by the **Na\'vi**, a tall, blue-skinned humanoid species who live in harmony with their environment. Humans have arrived on Pandora to mine a valuable mineral called **unobtanium**, which is crucial for solving Earth\'s energy crisis.\n\nTo interact with the Na\'vi and explore Pandora without succumbing to its toxic atmosphere, humans use **avatars** – genetically engineered Na\'vi bodies controlled remotely by human minds.\n\nThe protagonist is **Jake Sully**, a paraplegic former Marine who takes his deceased twin brother\'s place in the Avatar Program. Initially task

In [62]:
model_with_structure.invoke("Tell me about Avatar movie")

Movie(title='Avatar', year=2009, rating=7.9, genre='Science Fiction')

In [63]:
result = model_with_structure.invoke("Tell me about Avatar movie")
result

Movie(title='Avatar', year=2009, rating=7.8, genre='Science Fiction')

In [64]:
### message output alongside parsed structure

class Movie(BaseModel):
    """A movie with details"""
    title:str = Field(...,description="Title of the movie")
    year:int = Field(...,description="Year of release")
    rating:float = Field(...,description="Rating of the movie")
    genre:str = Field(...,description="Genre of the movie") 
    

model_with_structure = google_llm.with_structured_output(
    Movie,include_raw=True
)
model_with_structure


{
  raw: RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x0000014CC3675BD0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'description': 'A movie with details', 'properties': {'title': {'description': 'Title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'Year of release', 'title': 'Year', 'type': 'integer'}, 'rating': {'description': 'Rating of the movie', 'title': '

In [65]:
result = model_with_structure.invoke("Tell me about Avatar movie")
result

{'raw': AIMessage(content='{\n  "title": "Avatar",\n  "year": 2009,\n  "rating": 7.8,\n  "genre": "Science Fiction"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d01c2-d8b8-7403-9afa-1f25e92508bc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 41, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Avatar', year=2009, rating=7.8, genre='Science Fiction'),
 'parsing_error': None}

In [66]:
## nested structure

class Actor(BaseModel):
    """Actor details"""
    name:str = Field(...,description="Name of the actor")
    age:int = Field(...,description="Age of the actor")
    role:str = Field(...,description="Role of the actor")

class Movie(BaseModel):
    """A movie with details"""
    title:str = Field(...,description="Title of the movie")
    year:int = Field(...,description="Year of release")
    rating:float = Field(...,description="Rating of the movie")
    genre:str = Field(...,description="Genre of the movie") 
    actors:list[Actor] = Field(...,description="List of actors")

model_with_structure = google_llm.with_structured_output(
    Movie,include_raw=True
)
model_with_structure


{
  raw: RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x0000014CC3675BD0>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type': 'application/json', 'response_json_schema': {'$defs': {'Actor': {'description': 'Actor details', 'properties': {'name': {'description': 'Name of the actor', 'title': 'Name', 'type': 'string'}, 'age': {'description': 'Age of the actor', 'title': 'Age', 'type': 'integer'}, 'role': {'description': 'Role of the actor', 'titl

In [67]:
result = model_with_structure.invoke("Tell me about Avatar movie")
result

{'raw': AIMessage(content='{\n  "title": "Avatar",\n  "year": 2009,\n  "rating": 7.8,\n  "genre": "Science Fiction",\n  "actors": [\n    {\n      "name": "Sam Worthington",\n      "age": 47,\n      "role": "Jake Sully"\n    },\n    {\n      "name": "Zoe Saldana",\n      "age": 45,\n      "role": "Neytiri"\n    },\n    {\n      "name": "Sigourney Weaver",\n      "age": 74,\n      "role": "Dr. Grace Augustine"\n    }\n  ]\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d01c2-dcc8-7cf2-a463-7dbdf9eedde5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 158, 'total_tokens': 164, 'input_token_details': {'cache_read': 0}}),
 'parsed': Movie(title='Avatar', year=2009, rating=7.8, genre='Science Fiction', actors=[Actor(name='Sam Worthington', age=47, role='Jake Sully'), Actor(name='Zoe Saldana', age=45, role='Neytiri')

In [79]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

groq_llm = init_chat_model("groq:openai/gpt-oss-120b")
groq_llm


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=groq_llm,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='79fb580d-42d1-4873-b3d0-84cfed4a483f'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from text. According to system, we must output only JSON object matching ContactInfo schema. Must include name, email, phone. Provide compact JSON.\n\nWe need to ensure JSON is valid, compact (no whitespace). Provide fields name, email, phone.\n\nThus: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}\n\nReturn only that.'}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 238, 'total_tokens': 358, 'completion_time': 0.315794809, 'completion_tokens_details': {'reasoning_tokens': 88}, 'prompt_time': 0.02639099, 'prompt_tokens_details': None, 'queue_time': 0.1434821, 'total